# Corpus Processing
This notebook works through some basic audio corpus processing in Python.

## Step 1: Load Audio
Say we have a lot of audio files located within a directory and its subdirectories. We want to do some processing on all of them. First, we'll probe the directory and locate all audio files. (If you don't have a corpus, but you want to get one, you can use Python to download audio from Freesound.)

In [2]:
import os
import re

# Replace with your own directory name
directory = "D:\\Recording\\Samples\\granulation\\freesound_creative_commons_0"
audio_files = []

# Iterate over each subdirectory, and locate all audio files
for current_directory, _, files in os.walk(directory):
    for file in files:
        if re.search(r'(\.wav$)|(\.aif+$)', file, re.IGNORECASE):
            audio_files.append(os.path.join(current_directory, file))

There are several Python libraries for reading audio files. I recommend using `pedalboard`, a library developed at Spotify. Other libraries that can read audio files include `librosa`, `scipy`, and `soundfile`. The `pedalboard` library gives you granular control over reading audio files, and lets you read just part of a file at a time.

Be careful--this step will read *all* of the audio files into memory.

In [ ]:
import pedalboard as pb
audio = []
for file in audio_files:
    # We can't know all the sample rates, so we'll just resample all the audio to 44.1 kHz. You could upsample to 48 kHz if you want.
    with pb.io.AudioFile(file, 'r').resampled_to(44100) as audio_file:
        samples = audio_file.read(audio_file.frames)
        audio.append({
            "path": file,
            "samples": samples, # The audio samples, stored as a 2D NumPy array
            "sample_rate": audio_file.samplerate,
            "num_channels": audio_file.num_channels,
            "num_frames": audio_file.frames,
            "duration": audio_file.frames / audio_file.samplerate
        })

## Step 2: Process
In this example, we'll apply a highpass filter to remove any DC bias in the recordings, and we'll split each audio file into 2-second chunks with fade-in and fade-out at the end.

First we'll compute the filter coefficients using the `scipy` library. We'll use a cutoff frequency of 20 Hz. Best practice is to use "second-order sections," which divides a bigger filter into a series of smaller 2nd-order filters. Note that if we hadn't resampled all of the audio to the same sample rate, we'd want to compute the filter coefficients separately for each audio file.

In [ ]:
import scipy.signal

# Second-order sections coefficients for a 4th order Butterworth highpass filter with a cutoff of 20Hz
hpf = scipy.signal.butter(4, 20, 'high', output='sos', fs=44100)

Now we can iterate over all of the audio sample arrays and apply the filter.

In [5]:
for file in audio:
    file["samples"] = scipy.signal.sosfilt(hpf, file["samples"])

Now we want to split all of our audio into 2-second chunks. Of course, there might be a chunk shorter than 2 seconds at the end--we'll just discard that. Also, we'll apply a 0.5 second linear fade at the beginning and end, to avoid clicks.

In [ ]:
import numpy as np
CHUNK_SIZE = 44100 * 2

# A 0.5 second fade-in and fade-out
fade_in = np.linspace(0.0, 1.0, 44100//2)
fade_out = np.linspace(1.0, 0.0, 44100//2)

for file in audio:
    file["chunks"] = []
    for i in range(0, file["num_frames"], CHUNK_SIZE):
        if file["num_frames"] - i >= CHUNK_SIZE:
            chunk = file["samples"][i:i+CHUNK_SIZE]
            # Apply the fade
            chunk = np.hstack((
                chunk[:44100//2] * fade_in,  # We only fade in on the first 44100/2 samples
                chunk[44100//2:-44100//2],   # No fade applied to the central part of the audio
                chunk[-44100//2:] * fade_out # We fade out on the last 44100/2 samples
            ))
            file["chunks"].append(chunk)

## Step 3: Write the Audio
Last, we'll write the audio to a directory.